# Swin Transformer — Pneumonia Detection (Kaggle)

## Why Swin Transformer?
- **Local window attention** (7×7) instead of global → O(N) not O(N²)
- **Hierarchical stages** like a CNN — learns fine textures → mid-level → global patterns
- **4×4 patches** (vs ViT's 16×16) → finer initial resolution
- **Shifted windows** every other block → cross-window communication
- Works well on ~5K images from scratch (unlike plain ViT which needs millions)

## 1. Imports

In [2]:
import os, shutil, collections
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU") or "None — using CPU")

TensorFlow : 2.19.0
GPU        : None — using CPU


## 2. Dataset Paths (Kaggle)

In [ ]:
DATASET_PATH = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray'

train_path = os.path.join(DATASET_PATH, 'train')
val_path   = os.path.join(DATASET_PATH, 'val')
test_path  = os.path.join(DATASET_PATH, 'test')

print('Train exists:', os.path.exists(train_path))
print('Val exists:  ', os.path.exists(val_path))
print('Test exists: ', os.path.exists(test_path))

## 3. Configuration

In [ ]:
# ── Image ──────────────────────────────────────────────────────────────────────
IMAGE_SIZE   = (224, 224)
PATCH_SIZE   = 4           # 4×4 patches → 56×56 = 3136 tokens
WINDOW_SIZE  = 7           # local attention window

# ── Swin-T architecture ────────────────────────────────────────────────────────
EMBED_DIM    = 96
DEPTHS       = [2, 2, 6, 2]
NUM_HEADS    = [3, 6, 12, 24]
MLP_RATIO    = 4.0
DROPOUT      = 0.1
ATTN_DROPOUT = 0.1

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-4

# ── Paths ──────────────────────────────────────────────────────────────────────
MODEL_SAVE   = '/kaggle/working/swin_pneumonia.keras'
RESULTS_DIR  = '/kaggle/working/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Patch size   : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Window size  : {WINDOW_SIZE}×{WINDOW_SIZE}")
print(f"Stage depths : {DEPTHS}  →  total blocks: {sum(DEPTHS)}")

## 4. Data Generators

In [ ]:
# ── Merge tiny val/ (16 imgs) into combined_train/ ───────────────────────────
COMBINED_TRAIN = '/kaggle/working/combined_train'
for cls in ['NORMAL', 'PNEUMONIA']:
    dest = os.path.join(COMBINED_TRAIN, cls)
    os.makedirs(dest, exist_ok=True)
    for src_dir in [os.path.join(train_path, cls), os.path.join(val_path, cls)]:
        if os.path.isdir(src_dir):
            for fname in os.listdir(src_dir):
                src = os.path.join(src_dir, fname)
                dst = os.path.join(dest, fname)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    validation_split=0.10,
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    COMBINED_TRAIN,
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode='grayscale', class_mode='binary',
    subset='training', seed=42,
)
val_generator = train_datagen.flow_from_directory(
    COMBINED_TRAIN,
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode='grayscale', class_mode='binary',
    subset='validation', seed=42, shuffle=False,
)
test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    color_mode='grayscale', class_mode='binary', shuffle=False,
)

label_counts = collections.Counter(train_generator.classes)
total        = sum(label_counts.values())
class_weight = {cls: total / (2 * cnt) for cls, cnt in label_counts.items()}

print(f"Train : {train_generator.samples} | Val : {val_generator.samples} | Test : {test_generator.samples}")
print(f"Classes     : {train_generator.class_indices}")
print(f"Class weights: {class_weight}")

## 5. Swin Building Blocks

In [ ]:
import numpy as np
import cv2

# ── Pure-Python window helpers ────────────────────────────────────────────────
def window_partition(x, window_size):
    """(B,H,W,C) → (B*nW, ws, ws, C)"""
    B = tf.shape(x)[0]
    H, W, C = x.shape[1], x.shape[2], x.shape[3]
    x = tf.reshape(x, [B, H // window_size, window_size,
                           W // window_size, window_size, C])
    x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
    return tf.reshape(x, [-1, window_size, window_size, C])


def window_reverse(windows, window_size, H, W):
    """(B*nW, ws, ws, C) → (B, H, W, C)"""
    C = windows.shape[-1]
    x = tf.reshape(windows, [-1, H // window_size, W // window_size,
                                  window_size, window_size, C])
    x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
    return tf.reshape(x, [-1, H, W, C])


# ── Helper: build SW-MSA mask entirely in numpy ───────────────────────────────
def _build_shift_mask(H, W, window_size, shift_size):
    img_mask = np.zeros((H, W), dtype=np.float32)
    cnt = 0
    for hs in [slice(0, -window_size),
               slice(-window_size, -shift_size),
               slice(-shift_size, None)]:
        for ws in [slice(0, -window_size),
                   slice(-window_size, -shift_size),
                   slice(-shift_size, None)]:
            img_mask[hs, ws] = cnt
            cnt += 1

    # partition into windows manually in numpy
    nWh = H // window_size
    nWw = W // window_size
    m   = img_mask.reshape(nWh, window_size, nWw, window_size)
    m   = m.transpose(0, 2, 1, 3).reshape(-1, window_size * window_size)  # (nW, ws*ws)

    attn_mask = m[:, :, None] - m[:, None, :]          # (nW, ws*ws, ws*ws)
    attn_mask = np.where(attn_mask != 0, -100.0, 0.0).astype(np.float32)
    return attn_mask   # plain numpy array


# ── Helper: build relative position index in numpy ───────────────────────────
def _build_rel_pos_index(window_size):
    coords_h = np.arange(window_size)
    coords_w = np.arange(window_size)
    ch, cw   = np.meshgrid(coords_h, coords_w, indexing="ij")  # (Wh, Ww)
    ch = ch.reshape(-1);  cw = cw.reshape(-1)                   # (N,)
    rel_h = ch[:, None] - ch[None, :] + window_size - 1         # (N, N)
    rel_w = cw[:, None] - cw[None, :] + window_size - 1
    rel_index = rel_h * (2 * window_size - 1) + rel_w           # (N, N)
    return rel_index.astype(np.int32)                            # plain numpy


# ── WindowAttention ───────────────────────────────────────────────────────────
class WindowAttention(layers.Layer):
    def __init__(self, dim, window_size, num_heads,
                 attn_drop=0., proj_drop=0., **kwargs):
        super().__init__(**kwargs)
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads
        self.scale       = (dim // num_heads) ** -0.5

        self.qkv       = layers.Dense(dim * 3, use_bias=True)
        self.proj      = layers.Dense(dim)
        self.attn_drop = layers.Dropout(attn_drop)
        self.proj_drop = layers.Dropout(proj_drop)

        # ── numpy index — NEVER a tf.Tensor ──────────────────────────────────
        self._rel_idx = _build_rel_pos_index(window_size)   # (N, N) numpy int32

    def build(self, input_shape):
        table_size = (2 * self.window_size - 1) ** 2
        self.rel_pos_bias = self.add_weight(
            name="rel_pos_bias",
            shape=(table_size, self.num_heads),
            initializer="truncated_normal",
            trainable=True,
        )

    def call(self, x, mask=None, training=False):
        B_  = tf.shape(x)[0]
        N   = tf.shape(x)[1]
        C   = x.shape[-1]
        hd  = C // self.num_heads

        qkv = self.qkv(x)
        qkv = tf.reshape(qkv, [B_, N, 3, self.num_heads, hd])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])
        q, k, v = qkv[0] * self.scale, qkv[1], qkv[2]

        attn = tf.matmul(q, tf.transpose(k, [0, 1, 3, 2]))   # (B, heads, N, N)

        # Relative position bias — index is a plain numpy array → safe
        flat_idx = self._rel_idx.flatten()                    # numpy 1-D
        bias = tf.gather(self.rel_pos_bias, flat_idx)         # (N*N, heads)
        bias = tf.reshape(bias, [self.window_size**2,
                                  self.window_size**2,
                                  self.num_heads])
        bias = tf.transpose(bias, [2, 0, 1])[None]            # (1, heads, N, N)
        attn = attn + bias

        if mask is not None:
            nW   = tf.shape(mask)[0]
            attn = tf.reshape(attn, [B_ // nW, nW, self.num_heads, N, N])
            attn = attn + mask[None, :, None]                  # mask is a tf.constant
            attn = tf.reshape(attn, [-1, self.num_heads, N, N])

        attn = self.attn_drop(tf.nn.softmax(attn, axis=-1), training=training)
        x    = tf.transpose(tf.matmul(attn, v), [0, 2, 1, 3])
        x    = tf.reshape(x, [B_, N, C])
        return self.proj_drop(self.proj(x), training=training)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"dim": self.dim, "window_size": self.window_size,
                    "num_heads": self.num_heads})
        return cfg


# ── SwinTransformerBlock ──────────────────────────────────────────────────────
class SwinTransformerBlock(layers.Layer):
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., dropout=0., attn_dropout=0., **kwargs):
        super().__init__(**kwargs)
        self.dim              = dim
        self.input_resolution = input_resolution
        self.num_heads        = num_heads
        self.window_size      = window_size
        self.shift_size       = shift_size

        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.attn  = WindowAttention(dim, window_size, num_heads,
                                     attn_drop=attn_dropout, proj_drop=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = tf.keras.Sequential([
            layers.Dense(mlp_hidden, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(dim),
            layers.Dropout(dropout),
        ])

        # SW-MSA mask: pure numpy → tf.constant (safe across graph boundaries)
        if shift_size > 0:
            H, W = input_resolution
            mask_np = _build_shift_mask(H, W, window_size, shift_size)  # numpy
            self.attn_mask = tf.constant(mask_np, dtype=tf.float32)      # constant
        else:
            self.attn_mask = None

    def call(self, x, training=False):
        H, W = self.input_resolution
        B    = tf.shape(x)[0]
        C    = x.shape[-1]

        shortcut = x
        x = tf.reshape(self.norm1(x), [B, H, W, C])

        if self.shift_size > 0:
            x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1, 2])

        xw = window_partition(x, self.window_size)
        xw = tf.reshape(xw, [-1, self.window_size ** 2, C])
        xw = self.attn(xw, mask=self.attn_mask, training=training)

        x = window_reverse(
            tf.reshape(xw, [-1, self.window_size, self.window_size, C]),
            self.window_size, H, W)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[self.shift_size, self.shift_size], axis=[1, 2])

        x = shortcut + tf.reshape(x, [B, H * W, C])
        return x + self.mlp(self.norm2(x), training=training)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"dim": self.dim, "input_resolution": self.input_resolution,
                    "num_heads": self.num_heads, "window_size": self.window_size,
                    "shift_size": self.shift_size})
        return cfg


# ── PatchMerging ──────────────────────────────────────────────────────────────
class PatchMerging(layers.Layer):
    def __init__(self, input_resolution, dim, **kwargs):
        super().__init__(**kwargs)
        self.input_resolution = input_resolution
        self.dim       = dim
        self.norm      = layers.LayerNormalization(epsilon=1e-5)
        self.reduction = layers.Dense(2 * dim, use_bias=False)

    def call(self, x):
        H, W = self.input_resolution
        B = tf.shape(x)[0]
        x = tf.reshape(x, [B, H, W, -1])
        x = tf.concat([x[:, 0::2, 0::2, :], x[:, 1::2, 0::2, :],
                       x[:, 0::2, 1::2, :], x[:, 1::2, 1::2, :]], axis=-1)
        return self.reduction(self.norm(tf.reshape(x, [B, -1, 4 * self.dim])))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"input_resolution": self.input_resolution, "dim": self.dim})
        return cfg


# ── PatchEmbed ────────────────────────────────────────────────────────────────
class PatchEmbed(layers.Layer):
    def __init__(self, patch_size=4, embed_dim=96, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size,
                                  strides=patch_size, padding="valid")
        self.norm = layers.LayerNormalization(epsilon=1e-5)

    def call(self, x):
        x = self.proj(x)
        B, H, W, C = tf.shape(x)[0], x.shape[1], x.shape[2], x.shape[3]
        return self.norm(tf.reshape(x, [B, H * W, C]))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size})
        return cfg


print("All Swin building blocks defined — fully numpy-safe.")

## 6. Build Swin-T Model

In [ ]:
def build_swin(
    image_size=IMAGE_SIZE, patch_size=PATCH_SIZE, embed_dim=EMBED_DIM,
    depths=DEPTHS, num_heads=NUM_HEADS, window_size=WINDOW_SIZE,
    mlp_ratio=MLP_RATIO, dropout=DROPOUT, attn_dropout=ATTN_DROPOUT,
):
    H = W = image_size[0] // patch_size   # 56

    inp = layers.Input(shape=(*image_size, 1), name="xray_input")
    x   = PatchEmbed(patch_size, embed_dim, name="patch_embed")(inp)
    x   = layers.Dropout(dropout)(x)

    dim = embed_dim
    res = (H, W)

    for stage_idx, (depth, n_heads) in enumerate(zip(depths, num_heads)):
        for blk_idx in range(depth):
            shift = 0 if blk_idx % 2 == 0 else window_size // 2
            x = SwinTransformerBlock(
                dim=dim, input_resolution=res, num_heads=n_heads,
                window_size=window_size, shift_size=shift,
                mlp_ratio=mlp_ratio, dropout=dropout, attn_dropout=attn_dropout,
                name=f"stage{stage_idx}_block{blk_idx}",
            )(x)
        if stage_idx < len(depths) - 1:
            x   = PatchMerging(res, dim, name=f"patch_merge_{stage_idx}")(x)
            res = (res[0]//2, res[1]//2)
            dim = dim * 2

    x   = layers.LayerNormalization(epsilon=1e-5, name="final_norm")(x)
    x   = layers.GlobalAveragePooling1D(name="gap")(x)
    emb = layers.LayerNormalization(epsilon=1e-5, name="emb_norm")(x)
    x   = layers.Dense(256, activation="gelu", name="head_dense")(emb)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid", name="output")(x)

    return Model(inp, out, name="SwinT_Pneumonia"), Model(inp, emb, name="SwinT_Embeddings")


swin_model, embedding_model = build_swin()
swin_model.summary(line_length=90)

## 7. Compile & Train

In [ ]:
swin_model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=LR, weight_decay=0.05),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE, monitor="val_accuracy",
        save_best_only=True, verbose=1,
    ),
]

history = swin_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (tk, vk), title in zip(
    axes,
    [("accuracy","val_accuracy"), ("loss","val_loss")],
    ["Accuracy", "Loss"],
):
    ax.plot(history.history[tk], label="Train")
    ax.plot(history.history[vk], label="Val")
    ax.set_title(f"Swin-T — {title}")
    ax.set_xlabel("Epoch"); ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/swin_training_curves.png", dpi=150)
plt.show()

## 9. Evaluate on Test Set

In [ ]:
test_generator.reset()
loss, acc = swin_model.evaluate(test_generator, verbose=1)
print(f"\nTest Loss    : {loss:.4f}")
print(f"Test Accuracy: {acc:.4f}")

test_generator.reset()
y_true = test_generator.classes
y_pred = (swin_model.predict(test_generator, verbose=1).ravel() >= 0.5).astype(int)

print("\n── Classification Report ─────────────────────────────────────")
print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["NORMAL","PNEUMONIA"], yticklabels=["NORMAL","PNEUMONIA"])
plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.title("Confusion Matrix — Swin-T")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/swin_confusion_matrix.png", dpi=150)
plt.show()

## 10. t-SNE Embeddings

In [ ]:
from sklearn.manifold import TSNE

test_generator.reset()
embeddings = embedding_model.predict(test_generator, verbose=1)
labels     = test_generator.classes

coords = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000).fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
for cls, color, lbl in [(0,"#3B82F6","NORMAL"),(1,"#EF4444","PNEUMONIA")]:
    mask = labels == cls
    ax.scatter(coords[mask,0], coords[mask,1], c=color, label=lbl, alpha=0.6, s=20, linewidths=0)
ax.set_title("t-SNE of Swin-T Embeddings — Test Set", fontsize=13)
ax.set_xlabel("dim 1"); ax.set_ylabel("dim 2"); ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/swin_tsne_embeddings.png", dpi=150)
plt.show()

## 11. Feature Attention Visualisation

In [ ]:
import cv2
from tensorflow.keras.preprocessing.image import load_img, img_to_array


def get_swin_attention(model, img_array, stage=3, block=0):
    """
    Extract a spatial heatmap from a Swin block using a captured intermediate
    output. Works with Keras 3 where .output is not available on sub-layers.
    """
    # Run full model forward and capture the output of the target block
    # by rebuilding a sub-model up to (but not including) that block's MLP.
    # Strategy: collect outputs stage-by-stage using the named layers.
    captured = {}

    # We wrap the block's norm1 in a Lambda to capture its output
    block_name = f"stage{stage}_block{block}"

    # Build a list of all layers in order up to and including target block
    layer_names = [l.name for l in model.layers]
    if block_name not in layer_names:
        print(f"Layer {block_name} not found. Available: {layer_names}")
        return np.zeros((224, 224))

    # Use tf.GradientTape-free approach: run each layer manually
    x = img_array
    for layer in model.layers:
        if layer.name == "xray_input":
            continue
        try:
            x = layer(x, training=False)
        except TypeError:
            x = layer(x)
        if layer.name == block_name:
            # x here is the OUTPUT of the Swin block: (1, tokens, C)
            break

    # x is now (1, tokens, C) — use channel L2 norm as spatial importance
    tokens = x[0]                                    # (tokens, C)
    H = W  = int(tokens.shape[0] ** 0.5)
    feat   = tf.reshape(tokens, [H, W, tokens.shape[-1]])
    heat   = tf.norm(feat, axis=-1).numpy()          # (H, W)
    heat   = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    return cv2.resize(heat, (224, 224))


def plot_attention(model, img_array, true_label, ax_row):
    pred    = float(model.predict(img_array, verbose=0)[0][0])
    label   = "PNEUMONIA" if pred >= 0.5 else "NORMAL"
    heat    = get_swin_attention(model, img_array)
    img     = img_array[0, :, :, 0]
    img_rgb = np.stack([img] * 3, axis=-1)
    overlay = np.clip(0.55 * img_rgb + 0.45 * plt.cm.jet(heat)[:, :, :3], 0, 1)

    ax_row[0].imshow(img, cmap="gray");  ax_row[0].set_title(f"True: {true_label}"); ax_row[0].axis("off")
    ax_row[1].imshow(heat, cmap="jet");  ax_row[1].set_title("Feature Intensity");   ax_row[1].axis("off")
    ax_row[2].imshow(overlay);           ax_row[2].set_title(f"Pred: {label}");      ax_row[2].axis("off")


DATASET_PATH = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray'
test_path_viz = os.path.join(DATASET_PATH, 'test')

CLASSES = ["NORMAL", "PNEUMONIA"]
sample_paths = []
for cls in CLASSES:
    cls_dir = os.path.join(test_path_viz, cls)
    sample_paths += [(os.path.join(cls_dir, f), cls)
                     for f in sorted(os.listdir(cls_dir))[:2]]

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i, (fpath, true_cls) in enumerate(sample_paths):
    img = load_img(fpath, color_mode="grayscale", target_size=IMAGE_SIZE)
    arr = np.expand_dims(img_to_array(img) / 255.0, axis=0)
    plot_attention(swin_model, arr, true_cls, axes[i])

plt.suptitle("Swin-T Feature Maps — Test Samples", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/swin_attention_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Summary

In [ ]:
total_params = swin_model.count_params()

print("=" * 55)
print("  Swin-T Pneumonia — Summary")
print("=" * 55)
print(f"  Patch size        : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"  Window size       : {WINDOW_SIZE}×{WINDOW_SIZE}")
print(f"  Stage depths      : {DEPTHS}")
print(f"  Heads per stage   : {NUM_HEADS}")
print(f"  Base embed dim    : {EMBED_DIM}")
print(f"  Total parameters  : {total_params:,}")
print(f"  Test accuracy     : {acc:.4f}")
print("=" * 55)
print()
print("Outputs saved:")
print(f"  Model      → {MODEL_SAVE}")
print(f"  Curves     → {RESULTS_DIR}/swin_training_curves.png")
print(f"  Confusion  → {RESULTS_DIR}/swin_confusion_matrix.png")
print(f"  t-SNE      → {RESULTS_DIR}/swin_tsne_embeddings.png")
print(f"  Attention  → {RESULTS_DIR}/swin_attention_maps.png")